# Ablation Study & SOTA Comparison

Quantify the contribution of each modality and compare against published baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
from torch.utils.data import DataLoader

from vibropredict.data.enzyme_kinetics_dataset import EnzymeKineticsDataset
from vibropredict.models.vibropredict_hybrid import VibroPredictHybrid
from vibropredict.evaluation.ablation import run_ablation
from vibropredict.evaluation.visualization import (
    plot_correlation, plot_ablation_results, plot_gate_weights,
    plot_error_distribution,
)
from vibropredict.evaluation.sota_comparison import compare_with_baselines

## Load Trained Model & Test Data

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VibroPredictHybrid(fusion_dim=512, dropout=0.2).to(device)
checkpoint = torch.load('../checkpoints/vibropredict_full/best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_dataset = EnzymeKineticsDataset(
    csv_path='../data/vibropredict/splits/test.csv',
    vdos_dir='../data/vibropredict/vdos/',
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Run Ablation

In [ ]:
ablation_results = run_ablation(model, test_loader, device=device)
print(ablation_results.to_string(index=False))

In [ ]:
plot_ablation_results(ablation_results, save_path='../results/ablation_chart.png')

## SOTA Comparison

In [ ]:
our_full = ablation_results[ablation_results['variant'] == 'full'].iloc[0].to_dict()
comparison = compare_with_baselines(our_full)
print(comparison.to_string(index=False))

## Visualizations

In [ ]:
# Predictions vs targets scatter
# (Requires running model on test set first - see 07_model_training.ipynb)
# plot_correlation(predictions, targets, save_path='../results/correlation.png')
# plot_error_distribution(predictions, targets, save_path='../results/errors.png')
print('Run model inference first, then uncomment above lines.')